In [8]:
!pip install torch


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!pip install tensorflow



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
import pickle
import os

# 1. Load Data (Auto-Detect Columns)
raw_df = pd.read_csv("../data/archive.zip", encoding="latin-1")

# Inspect actual columns loaded
print("Loaded Columns:", raw_df.columns.tolist())

# Flexibly pick the text and label columns
if 'v1' in raw_df.columns and 'v2' in raw_df.columns:
    df = raw_df[['v1', 'v2']].copy()
    df.columns = ['label', 'message']
elif 'Category' in raw_df.columns and 'Message' in raw_df.columns:
    df = raw_df[['Category', 'Message']].copy()
    df.columns = ['label', 'message']
elif 'label' in raw_df.columns and 'text' in raw_df.columns:
    df = raw_df[['label', 'text']].copy()
    df.columns = ['label', 'message']
else:
    # Default to picking first two columns
    df = raw_df.iloc[:, [0, 1]].copy()
    df.columns = ['label', 'message']

# Map labels: ham -> 0, spam -> 1
df['label'] = df['label'].astype(str).str.lower()
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Handle potential missing/unmapped values
df = df.dropna(subset=['label_num', 'message'])

print("Cleaned Dataset Preview:")
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '../data/archive.zip'

In [7]:
# 2. Vocabulary & Tokenization Setup
def build_vocab(texts, max_words=10000):
    words = []
    for text in texts:
        words.extend(str(text).lower().split())
    counts = Counter(words)
    vocab = {word: i+2 for i, (word, _) in enumerate(counts.most_common(max_words-2))}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

vocab = build_vocab(df['message'])

def text_to_sequence(text, vocab, max_len=100):
    tokens = str(text).lower().split()
    seq = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    if len(seq) < max_len:
        seq += [vocab['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

X = np.array([text_to_sequence(msg, vocab) for msg in df['message']])
y = df['label_num'].values

# Split Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create PyTorch DataLoaders
class SpamDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SpamDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(SpamDataset(X_test, y_test), batch_size=32)

# 3. Define 1D CNN Architecture
class CNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=128, kernel_size=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1d = nn.Conv1d(embed_dim, num_filters, kernel_size)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(num_filters, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = self.relu(self.conv1d(x))
        x, _ = torch.max(x, dim=2)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.sigmoid(self.fc2(x))
        return x.squeeze()

# Initialize Model, Criterion, Optimizer
model = CNNTextClassifier(len(vocab))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 4. Train 1D CNN Model
print("Training CNN Model with PyTorch...")
model.train()
for epoch in range(5):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/5 - Loss: {total_loss/len(train_loader):.4f}")

# 5. Save Artifacts to models folder
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/cnn_pytorch_model.pth')
with open('../models/vocab.pickle', 'wb') as f:
    pickle.dump(vocab, f)

print("\nPyTorch CNN Model and Vocab saved successfully in '../models/' folder!")

Training CNN Model with PyTorch...
Epoch 1/5 - Loss: 0.3281
Epoch 2/5 - Loss: 0.1124
Epoch 3/5 - Loss: 0.0392
Epoch 4/5 - Loss: 0.0299
Epoch 5/5 - Loss: 0.0183

PyTorch CNN Model and Vocab saved successfully in '../models/' folder!
